# 07 — Deep Blue data: Bayesian single-time-point TIA and dose

Notebook 06 showed that one SPECT at about 100 h recovers organ TIAs within a few percent with fixed population kinetics, but gave point estimates only. Here the population uptake-washout shape (method 3 of notebook 06) becomes a **prior**, and one or two measurements update it: the result is a posterior TIA, and finally a mean absorbed dose in Gy, each with a credible interval.

**Model** (as in notebooks 02 and 06): $A(t) = A_0 \left(e^{-\lambda_w t} - e^{-\lambda_u t}\right)$, with $\lambda_w = \lambda_\mathrm{phys} + \lambda_\mathrm{bio}$, $\lambda_\mathrm{bio} \ge 0$, $\lambda_u > \lambda_w$, and $\mathrm{TIA} = A_0 \left(1/\lambda_w - 1/\lambda_u\right)$.

**Measurement model:** $\log A_k \sim \mathcal N(\log A(t_k), \sigma)$ with $\sigma = 0.10$ (10% relative error, the random term of the notebook 02 Monte Carlo).

**Estimators compared:**

| scheme | data | amplitude prior |
|---|---|---|
| Variant 1 (V1) | one SPECT, each of the four scans | flat |
| Variant 2 (V2) | scan 1 (3.7 h) as surrogate of an ideal pre-therapy PET + one SPECT at ~30 h or ~100 h | Gaussian on $\log A(3.7\,\mathrm h)$, width $\sigma_a$ |
| two time points (2TP) | SPECT at ~30 h + ~100 h | flat (Bayesian), or classical mono-exponential fit |

**Prior on the shape.** Three versions of the washout prior are compared (section 2):
- P0: the previous one, log-normal on $\lambda_w$ truncated at $\lambda_\mathrm{phys}$, independent of the uptake rate;
- P1: log-normal on $\lambda_\mathrm{bio}$;
- P2 (main): as P1, but with the tail of a Student t with 3 degrees of freedom on the slow side ($T_\mathrm{eff} \to T_\mathrm{phys}$).

For kidneys, P1 and P2 couple washout and uptake with the correlation of the Devasia fits. The uptake prior is generic, structure-specific or oracle; the last two are used only for the variant 2 ablation.

**Reference:** the 4-point uptake-washout TIA of notebook 02 with its Monte Carlo 68% interval (10% scenario), and the mean dose of the hybrid map of notebook 04.

**Steps**
1. Inputs.
2. Priors.
3. Posterior computation and MCMC check.
4. Variant 1 and the effect of the washout prior.
5. Variant 2 and the uptake-prior ablation.
6. Two-time-point baseline.
7. Comparison with notebook 06 vs SPECT time; 68% and 95% coverage.
8. Absorbed dose with intervals.
9. Findings and saving.

**About this notebook.** Bayesian single-time-point TIA and dose with literature priors; coverage.
- **Inputs:** nb02, nb04, nb06 outputs.
- **Outputs:** `data/processed/deepblue/kinetics/bayes_stp_*`.
- **Run time:** about 6 min.
- **Environment:** .venv.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from scipy import stats

WS = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(WS / "src"))
from paths import DATASET
PROC = WS / "data" / "processed" / "deepblue"
KIN = PROC / "kinetics"
PATIENTS = ["patient_4", "patient_6"]
SNMMI_LABEL = {"patient_4": "A", "patient_6": "B"}
SIGMA = 0.10
SIGMA_AMP = [0.10, 0.25, 0.50]
NU_T = 3
N_PRIOR = 400_000
SEED = 20261101
QS = [0.025, 0.16, 0.5, 0.84, 0.975]

TYPES = ["kidney", "spleen", "liver", "lesion"]
STRUCTURE_TYPE = {"ltkidney": "kidney", "rtkidney": "kidney", "spleen": "spleen", "liver_healthy": "liver"}


def structure_type(name):
    return "lesion" if name.startswith("lesion") else STRUCTURE_TYPE.get(name)


half_lives = {json.loads((PROC / p / "timing.json").read_text())["half_life_s"] for p in PATIENTS}
assert len(half_lives) == 1
T_PHYS_H = half_lives.pop() / 3600.0
LAM_PHYS = np.log(2) / T_PHYS_H

plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#52514e",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e4e3df", "grid.linewidth": 0.6})
print(f"T1/2 = {T_PHYS_H:.1f} h, lambda_phys = {LAM_PHYS:.5f} 1/h")

## 1. Inputs

The kinetic inputs are as in notebook 06:
- activities: `kinetics/tac.csv`;
- reference TIA: `fits_all_models.csv`, uptake-washout fit with `used == True`;
- reference 68% interval: `tia_uncertainty_mc.csv`, 10% scenario;
- notebook 06 single-time-point estimates: `stp_tia.csv`.

**Dose factors** come from notebook 04 (`organ_doses.csv`): the mean dose of the hybrid TIA map with VSV convolution, divided by the structure TIA. In the hybrid map each structure carries exactly the notebook 02 TIA, so this ratio is a Gy per MBq·h factor for that structure.

Using a single factor scales the whole dose, including the crossfire from neighbouring structures, with the structure's own TIA. For Lu-177 the self-dose dominates organ and lesion doses, so this approximation is acceptable for an interval. Notebook 05 found that VSV in water overestimates the Monte Carlo dose in the patient CT by 3-7% for these structures; the MC-CT factor is saved alongside.

In [ ]:
tac = pd.read_csv(KIN / "tac.csv")
tac["type"] = tac.structure.map(structure_type)
tac = tac[tac.type.notna()].copy()

fits = pd.read_csv(KIN / "fits_all_models.csv")
upt = fits[(fits.model == "uptake") & fits.structure.map(structure_type).notna()].set_index(["patient", "structure"])
assert upt.used.all()
unc = pd.read_csv(KIN / "tia_uncertainty_mc.csv")
unc = unc[unc.sigma_point == "10%"].set_index(["patient", "structure"])

ref = upt[["TIA_MBq_h", "T_eff_h", "T_uptake_h", "lam_bio_at_bound"]].rename(columns={"TIA_MBq_h": "TIA_ref"})
ref = ref.join(unc[["TIA_p16", "TIA_p84"]])
ref["type"] = [structure_type(s) for _, s in ref.index]
ref["lam_w"] = np.log(2) / ref.T_eff_h
ref["lam_u"] = np.log(2) / ref.T_uptake_h

doses = pd.read_csv(KIN / "organ_doses.csv").set_index(["patient", "structure"])
mc = pd.read_csv(KIN / "organ_doses_mc.csv")
mc = mc[mc["map"] == "hybrid"].set_index(["patient", "structure"])
ref["D_ref_Gy"] = doses.D_hybrid_VSV_Gy.reindex(ref.index)
ref["Gy_per_MBq_h"] = ref.D_ref_Gy / ref.TIA_ref
ref["Gy_per_MBq_h_MC_CT"] = mc.MC_CT.reindex(ref.index) / ref.TIA_ref

stp06 = pd.read_csv(KIN / "stp_tia.csv")
acts = tac.pivot_table(index=["patient", "structure"], columns="scan", values="activity_MBq")
times = tac.groupby(["patient", "scan"]).t_h.first().unstack()
display(times.round(2))
ref[["type", "TIA_ref", "D_ref_Gy", "Gy_per_MBq_h", "Gy_per_MBq_h_MC_CT"]].round(5)

## 2. Priors

**Sources** (verified and cited in notebook 06):
- Kidneys: Devasia TP et al., J Nucl Med 2021;62:1118-1125, Supplemental Table 1. Eq. 1 is the same model, with $k_e = \lambda_w$ and $k_a = \lambda_u$. Their patients 4 and 6 are our patients 4 and 6 and are excluded (notebook 06), leaving 16 of the 20 kidneys of the table.
- Other structures: Hänscheid H et al., J Nucl Med 2018;59:75-81, Results, section "Normalized Uptake Functions" (text). Median effective half-life (range) of the main component: livers 67 h (55-117), spleens 68 h (52-99), tumors 77 h (56-130).

**Washout.**
- P0 (previous version of this notebook): $\log\lambda_w \sim \mathcal N$, truncated at $\lambda_\mathrm{phys}$ by rejection.
  - Kidneys: median and standard deviation of $\log k_e$.
  - Others: median $\ln 2 / T_\mathrm{med}$, $\sigma = \ln(T_\max/T_\min)/3.92$.
- P1 and P2 parametrise $\lambda_\mathrm{bio} = \lambda_w - \lambda_\mathrm{phys}$, so $\lambda_\mathrm{bio} \ge 0$ needs no truncation.
  - Kidneys: $\lambda_\mathrm{bio} = k_e - \lambda_\mathrm{phys}$ for each Devasia kidney; location = log of the median, scale = standard deviation of the logs.
  - Others: location $\log(\ln 2 / T_\mathrm{med} - \lambda_\mathrm{phys})$, scale $\ln\!\big(\lambda_\mathrm{bio}(T_\min)/\lambda_\mathrm{bio}(T_\max)\big)/3.92$. This is the same **assumption** as before: the published range covers the central 95%.
  - P1 is log-normal.
  - P2 (split tail) is log-normal on the fast side and has a Student-t tail ($\nu = 3$) on the slow side, with the same location and scale. In standardised units $z = (\log\lambda_\mathrm{bio} - \mu)/\sigma$, it is half-normal for $z > 0$ and half-$t_3$ for $z < 0$, so the median stays at the published value.
  - A symmetric $t_3$ was tried first and rejected:
    - with a single late SPECT, a heavy tail toward fast washout makes the extrapolated TIA explode (95% half-width above 600% at 193 h);
    - with a flat prior on $A_0$ the posterior is improper: the weight $1/f_k \approx e^{\lambda_w t_k}$ has no finite mean under a heavy-tailed $\log\lambda_w$.
    - The heavy tail is needed only toward $T_\mathrm{phys}$.
- With either parametrisation, the prior median of $\lambda_w$ is the same as in notebook 06, so the plug-in estimate at the prior median shape is still method (3) of notebook 06.

**Kidney correlation.** In P1, $(\log\lambda_\mathrm{bio}, \log\lambda_u)$ is bivariate normal with the correlation of the Devasia fits computed on these same variables. Its value is printed below, next to the correlation on $(\log k_e, \log k_a)$, which is +0.51. In P2 the same correlation couples the two latent normals of a Gaussian copula; the washout marginal is the split-tail distribution. Liver, spleen and lesions have independent washout and uptake (no data on their correlation).

**Uptake.**
- *Generic* (main): the Devasia $k_a$ log-normal for all structures. For liver, spleen and lesions this is an **assumption**, since no DOTATATE value was found (notebook 06).
- *Structure-specific* (variant 2 ablation only), an **assumption**, not from the literature:
  - healthy liver fast: median $\lambda_u$ = 4 h⁻¹ ($T_\mathrm{uptake}$ ≈ 10 min), σ = 1.0;
  - lesions slow allowed: median 0.25 h⁻¹ ($T_\mathrm{uptake}$ ≈ 2.8 h), σ = 1.0;
  - kidneys and spleen: as the generic prior.

  These values were chosen after seeing the notebook 02 fits, so they are optimistic.
- *Oracle* (ablation only): log-normal centred on the structure's own 4-point $\lambda_u$ from notebook 02, σ = 0.2. It is an upper bound on what any uptake prior can do, not a usable method.

All priors are truncated to $\lambda_u > \lambda_w$ by rejection. Numerically, extreme heavy-tail draws are also rejected: $\lambda_w \ge 10$ h⁻¹ or $\lambda_u \ge 1000$ h⁻¹, i.e. $T_\mathrm{eff}$ < 4 min. The rejected fraction is reported.

**Choice of the main washout prior.** The criterion uses the prior alone: how much mass is left for $T_\mathrm{eff}$ near $T_\mathrm{phys}$, where 2 of the 6 lesions of notebook 02 lie (patient 4, 112 h and 160 h).

In [ ]:
DEVASIA_KE = np.array([0.013, 0.014, 0.014, 0.014, 0.015, 0.020, 0.010, 0.012,
                       0.010, 0.010, 0.014, 0.014, 0.012, 0.010, 0.007, 0.006])
DEVASIA_KA = np.array([0.488, 0.460, 0.535, 0.526, 2.817, 3.173, 1.330, 4.658,
                       1.925, 0.802, 0.713, 0.970, 0.462, 0.467, 0.227, 0.275])
HAENSCHEID_TEFF_H = {"liver": (67.0, 55.0, 117.0), "spleen": (68.0, 52.0, 99.0), "lesion": (77.0, 56.0, 130.0)}

lit06 = pd.read_csv(KIN / "stp_population_kinetics.csv")
lit06 = lit06[(lit06.source == "literature") & (lit06.type == "kidney")].iloc[0]
assert np.isclose(np.median(DEVASIA_KE), lit06.lam_w) and np.isclose(np.median(DEVASIA_KA), lit06.lam_u)


def lam_bio(t_eff):
    return np.log(2) / t_eff - LAM_PHYS


log_ka = np.log(DEVASIA_KA)
log_bio_k = np.log(DEVASIA_KE - LAM_PHYS)
CORR_KIDNEY = np.corrcoef(log_bio_k, log_ka)[0, 1]
UPTAKE = {"generic": {typ: (np.log(np.median(DEVASIA_KA)), log_ka.std(ddof=1)) for typ in TYPES}}
UPTAKE["specific"] = {**UPTAKE["generic"], "liver": (np.log(4.0), 1.0), "lesion": (np.log(0.25), 1.0)}
WASHOUT = {"lam_w": {"kidney": (np.log(np.median(DEVASIA_KE)), np.log(DEVASIA_KE).std(ddof=1))},
           "lam_bio": {"kidney": (np.log(np.median(DEVASIA_KE) - LAM_PHYS), log_bio_k.std(ddof=1))}}
for typ, (med, lo, hi) in HAENSCHEID_TEFF_H.items():
    WASHOUT["lam_w"][typ] = (np.log(np.log(2) / med), np.log(hi / lo) / 3.92)
    WASHOUT["lam_bio"][typ] = (np.log(lam_bio(med)), np.log(lam_bio(lo) / lam_bio(hi)) / 3.92)
PRIORS = {"P0 trunc lam_w": ("lam_w", "normal", False),
          "P1 lam_bio log-normal": ("lam_bio", "normal", True),
          "P2 lam_bio split-t3": ("lam_bio", "t", True)}
MAIN = "P2 lam_bio split-t3"
print(f"Devasia kidneys: corr(log ke, log ka) = {np.corrcoef(np.log(DEVASIA_KE), log_ka)[0, 1]:+.3f}, "
      f"corr(log lam_bio, log ka) = {CORR_KIDNEY:+.3f}")

def split_tail(n):
    return np.where(n < 0, stats.t.ppf(np.clip(stats.norm.cdf(n), 1e-300, None), NU_T), n)


rng = np.random.default_rng(SEED)
_cache = {}


def sample_prior(prior, typ, uptake="generic", oracle_lu=None, n=N_PRIOR):
    key = (prior, typ, uptake, oracle_lu)
    if key in _cache:
        return _cache[key]
    par, tail, correlated = PRIORS[prior]
    (mw, sw) = WASHOUT[par][typ]
    (mu, su) = (np.log(oracle_lu), 0.2) if uptake == "oracle" else UPTAKE[uptake][typ]
    rho = CORR_KIDNEY if (typ == "kidney" and correlated and uptake != "oracle") else 0.0
    out, total, kept = [], 0, 0
    while kept < n:
        m = 2 * n
        z = rng.standard_normal((m, 2))
        z[:, 1] = rho * z[:, 0] + np.sqrt(1 - rho ** 2) * z[:, 1]
        if tail == "t":
            z[:, 0] = split_tail(z[:, 0])
        x = np.exp(mw + sw * z[:, 0])
        lw = x if par == "lam_w" else LAM_PHYS + x
        lu = np.exp(mu + su * z[:, 1])
        ok = (lw >= LAM_PHYS) & (lu > lw) & (lw < 10.0) & (lu < 1e3)
        out.append(np.column_stack([lw[ok], lu[ok]]))
        total, kept = total + m, kept + ok.sum()
    s = np.concatenate(out)[:n]
    _cache[key] = (s[:, 0], s[:, 1], 1 - kept / total)
    return _cache[key]


rows = []
for prior in PRIORS:
    for typ in TYPES:
        lw, lu, rejected = sample_prior(prior, typ)
        t_eff = np.log(2) / lw
        rows.append({"prior": prior, "type": typ, "rejected %": 100 * rejected,
                     "T_eff median (h)": np.median(t_eff),
                     "T_eff 2.5-97.5% (h)": "-".join(f"{x:.0f}" for x in np.percentile(t_eff, [2.5, 97.5])),
                     "P(T_eff > 112 h) %": 100 * np.mean(t_eff > 112), "P(T_eff > 130 h) %": 100 * np.mean(t_eff > 130),
                     "P(T_eff > 150 h) %": 100 * np.mean(t_eff > 150)})
prior_table = pd.DataFrame(rows).set_index(["prior", "type"])
prior_table.round(2)

**Choice of the main prior.** For lesions, P1 (log-normal on $\lambda_\mathrm{bio}$) leaves *less* mass than P0 above 130 h. Parametrising on $\lambda_\mathrm{bio}$ alone does not solve the problem: in log space, $T_\mathrm{eff} \to T_\mathrm{phys}$ means $\log\lambda_\mathrm{bio} \to -\infty$, which a Gaussian tail suppresses. The slow-side t tail of P2 keeps a few percent of the mass above 130 h and still about 0.7% above 150 h, while leaving the fast side as in P1. **P2 is the main prior**; P0 and P1 are kept for comparison.

The position of the notebook 02 fits in each prior is shown below as standardised distances. `P2 tail prob` is the prior probability of a $\lambda_\mathrm{bio}$ at least that far from the median on the same side. It uses the $t_3$ tail on the slow side and the normal tail on the fast side.

In [ ]:
rows = []
for (patient, structure), r in ref.iterrows():
    typ = r.type
    row = {"patient": patient, "structure": structure, "T_eff_h": r.T_eff_h, "lam_u": r.lam_u}
    mw, sw = WASHOUT["lam_w"][typ]
    row["z P0 lam_w"] = (np.log(r.lam_w) - mw) / sw
    mb, sb = WASHOUT["lam_bio"][typ]
    lb = max(r.lam_w - LAM_PHYS, 1e-6)
    row["z P1/P2 lam_bio"] = (np.log(lb) - mb) / sb
    z = row["z P1/P2 lam_bio"]
    row["P2 tail prob"] = stats.t.cdf(z, NU_T) if z < 0 else stats.norm.sf(z)
    mu, su = UPTAKE["generic"][typ]
    row["z generic lam_u"] = (np.log(r.lam_u) - mu) / su
    mu, su = UPTAKE["specific"][typ]
    row["z specific lam_u"] = (np.log(r.lam_u) - mu) / su
    rows.append(row)
pd.DataFrame(rows).set_index(["patient", "structure"]).round(3)

Patient 4 lesion2 has $\lambda_\mathrm{bio}$ at the bound in notebook 02 (set to $10^{-6}$ h⁻¹ here): it is extreme under any prior that is centred on the published lesion half-lives.

## 3. Posterior

The parameters are $(A_0, \lambda_w, \lambda_u)$, with measurements $(t_k, A_k, \sigma_k)$. Notation:
- $f_k = e^{-\lambda_w t_k} - e^{-\lambda_u t_k}$;
- $m_k = \log A_k - \log f_k$;
- $\tau_k = 1/\sigma_k^2$.

For a fixed shape the model is linear in $\log A_0$. With a flat prior on $\log A_0$:
- $\log A_0 \mid \text{shape} \sim \mathcal N(\bar m, 1/\sqrt{\textstyle\sum\tau_k})$, with $\bar m$ the $\tau$-weighted mean of the $m_k$;
- the marginal weight of the shape is $w = \exp\!\big(-\tfrac12 \sum_k \tau_k (m_k - \bar m)^2\big)$.

The other amplitude priors follow from this case:
- Variant 2's amplitude prior on $\log A(3.7\,\mathrm h)$ is exactly an extra measurement at 3.7 h with $\sigma_k = \sigma_a$.
- A flat prior on $A_0$ (linear) multiplies the weight by $e^{\bar m}$ and shifts the conditional mean by $1/\sum\tau_k$.

The shape is sampled by importance sampling with the prior as proposal (400,000 draws per prior and structure type). Posterior quantiles (2.5, 16, 50, 84, 97.5%) are weighted quantiles of $\log\mathrm{TIA} = \log A_0 + \log(1/\lambda_w - 1/\lambda_u)$, with one Gaussian draw of $\log A_0$ per sample. The effective sample size $(\sum w)^2/\sum w^2$ is reported.

**No partial-volume term.** The estimate and the reference use the same VOI and the same activities, so a recovery error cancels in the comparison. For the absolute uncertainty of a lesion TIA or dose, add 10-20% as in notebook 02.

In [ ]:
def wquantile(x, w, q):
    i = np.argsort(x)
    c = np.cumsum(w[i])
    return np.interp(np.asarray(q) * c[-1], c, x[i])


def log_f(lw, lu, t):
    return -lw * t + np.log(-np.expm1(-(lu - lw) * t))


def posterior(lw, lu, meas, a0_prior="log"):
    g = 1 / lw - 1 / lu
    m = np.array([np.log(a) - log_f(lw, lu, t) for t, a, _ in meas])
    tau = np.array([1 / s ** 2 for _, _, s in meas])[:, None]
    big_t = tau.sum()
    mbar = (tau * m).sum(0) / big_t
    logw = -0.5 * (tau * (m - mbar) ** 2).sum(0)
    if a0_prior == "flat":
        logw, mbar = logw + mbar, mbar + 1 / big_t
    w = np.exp(logw - logw.max())
    log_tia = mbar + rng.standard_normal(len(mbar)) / np.sqrt(big_t) + np.log(g)
    q = np.exp(wquantile(log_tia, w, QS))
    return {"TIA_p2.5": q[0], "TIA_p16": q[1], "TIA_median": q[2], "TIA_p84": q[3], "TIA_p97.5": q[4],
            "ess": w.sum() ** 2 / (w ** 2).sum(),
            "T_eff_post_h": np.log(2) / wquantile(lw, w, 0.5), "lam_u_post": wquantile(lu, w, 0.5)}

**MCMC check.** A random-walk Metropolis sampler on $\log A_0$ and the two latent normals of the prior uses the full posterior under the main prior P2 (including the copula for kidneys), with no analytic step. The latent normals map deterministically to $\lambda_\mathrm{bio}$ (split tail) and $\lambda_u$. Three cases:
- patient 6 right kidney, scan 3 alone;
- patient 4 lesion2, scan 3 alone (the heavy-tail case);
- patient 6 lesion2, scans 1 + 2 with $\sigma_a$ = 10%.

In [ ]:
def latent_to_rates(n1, n2, typ):
    mb, sb = WASHOUT["lam_bio"][typ]
    mu, su = UPTAKE["generic"][typ]
    r = CORR_KIDNEY if typ == "kidney" else 0.0
    zb = split_tail(np.array([n1]))[0]
    zu = r * n1 + np.sqrt(1 - r ** 2) * n2
    return LAM_PHYS + np.exp(mb + sb * zb), np.exp(mu + su * zu)


def metropolis(typ, meas, n=400_000, burn=40_000, step=(0.08, 0.5, 0.5)):
    r = np.random.default_rng(1)
    t = np.array([m[0] for m in meas])
    a = np.array([m[1] for m in meas])
    s = np.array([m[2] for m in meas])

    def lp(th):
        la0, n1, n2 = th
        lw, lu = latent_to_rates(n1, n2, typ)
        if lu <= lw:
            return -np.inf
        if lw >= 10.0 or lu >= 1e3:
            return -np.inf
        pred = la0 + log_f(lw, lu, t)
        return -0.5 * (n1 ** 2 + n2 ** 2) - 0.5 * np.sum(((np.log(a) - pred) / s) ** 2)

    lw0, lu0 = latent_to_rates(0.0, 0.0, typ)
    th = np.array([np.log(a[-1]) - log_f(lw0, lu0, t[-1]), 0.0, 0.0])
    cur, chain, acc = lp(th), np.empty(n), 0
    steps = r.standard_normal((n, 3)) * np.array(step)
    u = np.log(r.random(n))
    for i in range(n):
        prop = th + steps[i]
        lpp = lp(prop)
        if u[i] < lpp - cur:
            th, cur, acc = prop, lpp, acc + 1
        lw, lu = latent_to_rates(th[1], th[2], typ)
        chain[i] = np.exp(th[0]) * (1 / lw - 1 / lu)
    return np.percentile(chain[burn:], [2.5, 16, 50, 84, 97.5]), acc / n


def measurements(patient, structure, scans, sigma_first=SIGMA):
    return [(times.loc[patient, sc], acts.loc[(patient, structure), sc], sigma_first if i == 0 else SIGMA)
            for i, sc in enumerate(scans)]


checks = []
for patient, structure, scans, s0 in [("patient_6", "rtkidney", ["scan3"], SIGMA),
                                       ("patient_4", "lesion2", ["scan3"], SIGMA),
                                       ("patient_6", "lesion2", ["scan1", "scan2"], 0.10)]:
    typ = structure_type(structure)
    meas = measurements(patient, structure, scans, s0)
    q_mc, acc = metropolis(typ, meas)
    lw, lu, _ = sample_prior(MAIN, typ)
    p = posterior(lw, lu, meas)
    checks.append({"case": f"{patient} {structure} {'+'.join(scans)}",
                   "MCMC p2.5/16/50/84/97.5": np.round(q_mc, 0),
                   "IS p2.5/16/50/84/97.5": np.round([p[k] for k in ["TIA_p2.5", "TIA_p16", "TIA_median", "TIA_p84", "TIA_p97.5"]], 0),
                   "acceptance": round(acc, 2), "IS ess": round(p["ess"])})
pd.DataFrame(checks)

## 4–6. All estimates

The cell below computes every scheme for each structure:
- **V1**: each scan alone with a flat prior on $\log A_0$, under the three washout priors. Also the flat prior on $A_0$ under P0 (under P1 and P2 it is improper or nearly so), and the structure-specific and oracle uptake priors under P2 at scans 2-3 (the paired comparison for the ablation).
- **V2**: scan 1 + scan 2 or 3, under P2 with $\sigma_a$ = 10, 25 and 50%; under P0 with $\sigma_a$ = 10% (the previous version); and the ablation with $\sigma_a$ = 10% under the structure-specific and oracle uptake priors.
- **2TP**: scans 2 + 3, Bayesian under P2 with a flat prior on $\log A_0$. Classical baseline: a mono-exponential through the two points, $\lambda = \ln(A_2/A_3)/(t_3 - t_2)$ with $\lambda \ge \lambda_\mathrm{phys}$, integrated from 0 to ∞.

In [ ]:
def tia_uptake(a, t, lam_w, lam_u):
    return a * (1 / lam_w - 1 / lam_u) / (np.exp(-lam_w * t) - np.exp(-lam_u * t))


def run(label, scheme, prior, patient, structure, scans, s0=SIGMA, uptake="generic", a0_prior="log", sigma_amp=np.nan):
    typ = structure_type(structure)
    oracle = ref.loc[(patient, structure), "lam_u"] if uptake == "oracle" else None
    lw, lu, _ = sample_prior(prior, typ, uptake, oracle)
    p = posterior(lw, lu, measurements(patient, structure, scans, s0), a0_prior)
    return {"label": label, "scheme": scheme, "prior": prior, "uptake": uptake, "a0_prior": a0_prior,
            "sigma_amp": sigma_amp, "patient": patient, "structure": structure, "type": typ,
            "scans": "+".join(scans), "scan": scans[-1], "t_spect_h": times.loc[patient, scans[-1]], **p}


rows = []
for (patient, structure), r in ref.iterrows():
    for scan in times.columns:
        for prior in PRIORS:
            rows.append(run(f"V1 [{prior}]", "V1", prior, patient, structure, [scan]))
        rows.append(run("V1 [P0 trunc lam_w, flat A0]", "V1", "P0 trunc lam_w", patient, structure, [scan], a0_prior="flat"))
    for scan in ["scan2", "scan3"]:
        for uptake in ["specific", "oracle"]:
            rows.append(run(f"V1 [{MAIN}, {uptake} uptake]", "V1", MAIN, patient, structure, [scan], uptake=uptake))
        for sa in SIGMA_AMP:
            rows.append(run(f"V2 sa={sa:.2f} [{MAIN}]", "V2", MAIN, patient, structure, ["scan1", scan], sa, sigma_amp=sa))
        rows.append(run("V2 sa=0.10 [P0 trunc lam_w]", "V2", "P0 trunc lam_w", patient, structure, ["scan1", scan], 0.10,
                        sigma_amp=0.10))
        for uptake in ["specific", "oracle"]:
            rows.append(run(f"V2 sa=0.10 [{MAIN}, {uptake} uptake]", "V2", MAIN, patient, structure, ["scan1", scan], 0.10,
                            uptake=uptake, sigma_amp=0.10))
    rows.append(run(f"2TP Bayes [{MAIN}]", "2TP", MAIN, patient, structure, ["scan2", "scan3"]))
    t2, t3 = times.loc[patient, ["scan2", "scan3"]]
    a2, a3 = acts.loc[(patient, structure), ["scan2", "scan3"]]
    lam = max(np.log(a2 / a3) / (t3 - t2), LAM_PHYS)
    rows.append({"label": "2TP classical mono", "scheme": "2TP", "prior": "none", "patient": patient,
                 "structure": structure, "type": r.type, "scans": "scan2+scan3", "scan": "scan3",
                 "t_spect_h": t3, "TIA_median": a2 * np.exp(lam * t2) / lam, "T_eff_post_h": np.log(2) / lam})

bay = pd.DataFrame(rows).join(ref[["TIA_ref", "TIA_p16", "TIA_p84", "T_eff_h"]].rename(
    columns={"TIA_p16": "ref_p16", "TIA_p84": "ref_p84", "T_eff_h": "T_eff_ref_h"}), on=["patient", "structure"])
bay["error"] = bay.TIA_median / bay.TIA_ref - 1
bay["half_width_68_%"] = 100 * (bay.TIA_p84 - bay.TIA_p16) / 2 / bay.TIA_median
bay["half_width_95_%"] = 100 * (bay["TIA_p97.5"] - bay["TIA_p2.5"]) / 2 / bay.TIA_median
has_int = bay.TIA_p16.notna()
bay["ref_in_post68"] = ((bay.TIA_ref >= bay.TIA_p16) & (bay.TIA_ref <= bay.TIA_p84)).where(has_int)
bay["ref_in_post95"] = ((bay.TIA_ref >= bay["TIA_p2.5"]) & (bay.TIA_ref <= bay["TIA_p97.5"])).where(has_int)
bay["est_in_ref68"] = bay.TIA_median.between(bay.ref_p16, bay.ref_p84)
print(f"{len(bay)} estimates; minimum effective sample size {bay.ess.min():.0f} "
      f"({bay.loc[bay.ess.idxmin(), 'label']}, {bay.loc[bay.ess.idxmin(), 'structure']})")

In [ ]:
def summary(g):
    e = g.error.abs()
    c68, c95 = g.ref_in_post68.dropna(), g.ref_in_post95.dropna()
    frac = lambda c: f"{int(c.sum())}/{len(c)}" if len(c) else "-"
    return pd.Series({"n": len(g), "median |error| %": 100 * e.median(), "max |error| %": 100 * e.max(),
                      "median error %": 100 * g.error.median(),
                      "half-width 68 %": g["half_width_68_%"].median(), "half-width 95 %": g["half_width_95_%"].median(),
                      "coverage 68": frac(c68), "coverage 95": frac(c95), "est in ref 68": f"{int(g.est_in_ref68.sum())}/{len(g)}"})

### 4. Variant 1 and the effect of the washout prior

**Consistency with notebook 06.**
- For kidneys, the plug-in estimate at the prior median shape is identical to method (3) of notebook 06; the prior median of $\lambda_w$ does not depend on the parametrisation.
- The table gives the ratio of the posterior median to that plug-in, under P2 with a flat prior on $\log A_0$ and under P0 with a flat prior on $A_0$ (the first version of this notebook). The flat prior on $A_0$ weights the shape by $1/f_k$ and biases the TIA upward.

In [ ]:
v1 = bay[bay.scheme == "V1"].copy()
med = {typ: (LAM_PHYS + np.exp(WASHOUT["lam_bio"][typ][0]), np.exp(UPTAKE["generic"][typ][0])) for typ in TYPES}
v1["TIA_plugin"] = [tia_uptake(acts.loc[(r.patient, r.structure), r.scan], r.t_spect_h, *med[r.type]) for r in v1.itertuples()]
k06 = stp06[(stp06.source == "literature") & (stp06.method == "uptake-washout, population shape")].set_index(
    ["patient", "structure", "scan"]).TIA_stp
kid = v1[(v1.type == "kidney") & (v1.label == f"V1 [{MAIN}]")].set_index(["patient", "structure", "scan"])
assert np.allclose(kid.TIA_plugin, k06.reindex(kid.index))
print("kidney plug-in = notebook 06 method (3), literature: identical")
v1["post_over_plugin"] = v1.TIA_median / v1.TIA_plugin
v1[v1.label.isin([f"V1 [{MAIN}]", "V1 [P0 trunc lam_w, flat A0]"])].groupby(["label", "scan"]).post_over_plugin.agg(
    ["min", "median", "max"]).unstack("scan").round(3)

In [ ]:
V1_PRIORS = [f"V1 [{p}]" for p in PRIORS] + ["V1 [P0 trunc lam_w, flat A0]"]
v1.groupby(["label", "scan"]).apply(summary, include_groups=False).reindex(V1_PRIORS, level=0).round(1)

**Patient 4 lesions under the three washout priors.** These are the structures with $T_\mathrm{eff}$ near $T_\mathrm{phys}$. The table gives the error (%) and the posterior $T_\mathrm{eff}$ for each scan.

In [ ]:
p4l = v1[(v1.patient == "patient_4") & v1.structure.isin(["lesion1", "lesion2"]) & v1.label.isin([f"V1 [{p}]" for p in PRIORS])]
t1 = p4l.assign(error=100 * p4l.error).pivot_table(index=["structure", "label"], columns="scan",
                                                    values=["error", "T_eff_post_h", "ref_in_post68"])
t1.round(1)

### 5. Variant 2 and the uptake-prior ablation

Two comparisons:
- **Amplitude prior width:** $\sigma_a$ = 10, 25 and 50% under P2, plus P0 with $\sigma_a$ = 10% (the previous version).
- **Ablation:** for each uptake prior (generic, structure-specific, oracle), variant 2 is compared with variant 1 *under the same prior*, at the same SPECT time. The **gain** is the reduction of the median |error| when the 3.7 h amplitude is added.
  - If the gain becomes clearly positive with the structure-specific or oracle uptake prior, the earlier failure was prior misspecification.
  - If it stays small or negative even with the oracle, the limitation is intrinsic: an early point in the uptake phase carries little information on the TIA.

In [ ]:
V2_LABELS = [f"V2 sa={sa:.2f} [{MAIN}]" for sa in SIGMA_AMP] + ["V2 sa=0.10 [P0 trunc lam_w]"]
bay[bay.label.isin(V2_LABELS)].groupby(["label", "scan"]).apply(summary, include_groups=False).reindex(V2_LABELS, level=0).round(1)

In [ ]:
pairs = {"generic": (f"V1 [{MAIN}]", f"V2 sa=0.10 [{MAIN}]"),
         "specific": (f"V1 [{MAIN}, specific uptake]", f"V2 sa=0.10 [{MAIN}, specific uptake]"),
         "oracle": (f"V1 [{MAIN}, oracle uptake]", f"V2 sa=0.10 [{MAIN}, oracle uptake]")}
rows = []
for uptake, (l1, l2) in pairs.items():
    for scan in ["scan2", "scan3"]:
        for group, sel in [("all", lambda d: d), ("liver + lesions", lambda d: d[d.type.isin(["liver", "lesion"])]),
                           ("kidneys", lambda d: d[d.type == "kidney"])]:
            a = sel(bay[(bay.label == l1) & (bay.scan == scan)])
            b = sel(bay[(bay.label == l2) & (bay.scan == scan)])
            rows.append({"uptake prior": uptake, "SPECT": scan, "structures": group, "n": len(a),
                         "V1 median |err| %": 100 * a.error.abs().median(), "V2 median |err| %": 100 * b.error.abs().median(),
                         "V1 max %": 100 * a.error.abs().max(), "V2 max %": 100 * b.error.abs().max(),
                         "V1 half-width 68 %": a["half_width_68_%"].median(), "V2 half-width 68 %": b["half_width_68_%"].median(),
                         "V1 cov68": f"{int(a.ref_in_post68.sum())}/{len(a)}", "V2 cov68": f"{int(b.ref_in_post68.sum())}/{len(b)}"})
ablation = pd.DataFrame(rows)
ablation["gain (V1 - V2) %"] = ablation["V1 median |err| %"] - ablation["V2 median |err| %"]
ablation.set_index(["uptake prior", "SPECT", "structures"]).round(1)

In [ ]:
sel = bay[bay.label.isin([pairs[u][1] for u in pairs]) & bay.type.isin(["kidney", "liver", "lesion"])]
piv = sel.assign(error=100 * sel.error).pivot_table(index=["patient", "structure", "scan"], columns="uptake",
                                                    values=["error", "T_eff_post_h"])
piv.columns = [f"V2 {a} ({b})" for a, b in piv.columns]
piv.join(ref[["T_eff_h", "lam_u"]].rename(columns={"T_eff_h": "ref T_eff_h", "lam_u": "ref lam_u"}),
         on=["patient", "structure"]).round(2)

### 6. Two-time-point baseline (SPECT at ~30 h + ~100 h)

In [ ]:
TP2 = [f"2TP Bayes [{MAIN}]", "2TP classical mono"]
t2tab = bay[bay.label.isin(TP2)].assign(error=lambda d: 100 * d.error).pivot_table(
    index=["patient", "structure"], columns="label", values=["error", "T_eff_post_h", "half_width_68_%"])
display(t2tab.round(1))
bay[bay.label.isin(TP2)].groupby("label").apply(summary, include_groups=False).round(1)

## 7. Comparison with notebook 06 vs SPECT time; 68% and 95% coverage

Main estimators, with notebook 06 methods using literature kinetics (no interval):
- "coverage 68" and "coverage 95": number of structures whose 4-point reference TIA falls inside the posterior 68% or 95% interval;
- "est in ref 68": number of point estimates inside the reference 68% interval (the criterion of notebook 06).

With 13 structures, the binomial 1 σ is about ±1.7 structures at 68% and ±0.8 at 95%. The reference is itself uncertain (organs 5-9%, lesions 11-24%) and shares the SPECT point with the estimate.

In [ ]:
METHODS06 = {"nb06 mono, literature T_eff": ("mono, population T_eff", "literature"),
             "nb06 Hänscheid": ("Hänscheid 2t/ln2", "none"),
             "nb06 uptake-washout, literature (kidneys)": ("uptake-washout, population shape", "literature")}
r06 = []
for name, (method, source) in METHODS06.items():
    s = stp06[(stp06.method == method) & (stp06.source == source)]
    r06.append(pd.DataFrame({"label": name, "patient": s.patient, "structure": s.structure, "type": s.type, "scan": s.scan,
                             "t_spect_h": s.t_h, "TIA_median": s.TIA_stp, "TIA_ref": s.TIA_ref, "error": s.error,
                             "est_in_ref68": s.in_68}))
allm = pd.concat([*r06, bay], ignore_index=True)
MAIN_LABELS = list(METHODS06) + [f"V1 [{MAIN}]", f"V2 sa=0.10 [{MAIN}]", f"2TP Bayes [{MAIN}]", "2TP classical mono"]
comp = allm[allm.label.isin(MAIN_LABELS)].groupby(["label", "scan"]).apply(summary, include_groups=False).reindex(MAIN_LABELS, level=0)
comp.round(1)

In [ ]:
cov = bay[bay.TIA_p16.notna()].groupby(["label", "scan"]).apply(summary, include_groups=False)[
    ["median |error| %", "half-width 68 %", "half-width 95 %", "coverage 68", "coverage 95"]]
cov.round(1)

**Error vs SPECT time.** One column per structure type.
- Top row: variant 1 under P2. Bottom row: variant 2 with $\sigma_a$ = 10% under P2, placed at the time of the SPECT.
- Thick bars: posterior 68% interval; thin bars: 95% interval. Filled marker: the 68% interval contains the reference.
- Orange: the best notebook 06 literature method for the structure type (uptake-washout shape for kidneys, mono $T_\mathrm{eff}$ otherwise).
- Grey band: median reference 68% interval of the type.
- Points are shifted slightly in time to separate structures; the y-axis is clipped at ±60%.

In [ ]:
C_BAY, C_06 = "#2a78d6", "#eb6834"
MARKERS = {"patient_4": "o", "patient_6": "s"}
CLIP = 0.6
ref["rel_p16"] = ref.TIA_p16 / ref.TIA_ref - 1
ref["rel_p84"] = ref.TIA_p84 / ref.TIA_ref - 1
best06 = {"kidney": "nb06 uptake-washout, literature (kidneys)"}
ROWS = [(f"V1 [{MAIN}]", "variant 1: SPECT alone"), (f"V2 sa=0.10 [{MAIN}]", "variant 2: 3.7 h amplitude\n($\\sigma_a$ = 10%) + SPECT")]


def rel(x, r):
    return 100 * (x / r - 1).clip(-CLIP, CLIP)


fig, axes = plt.subplots(2, len(TYPES), figsize=(14, 7.5), sharex=True, sharey=True)
for j, typ in enumerate(TYPES):
    rt = ref[ref.type == typ]
    structs = list(rt.index)
    for i, (label, title) in enumerate(ROWS):
        ax = axes[i, j]
        ax.axhspan(100 * rt.rel_p16.median(), 100 * rt.rel_p84.median(), color="#d9d8d3", alpha=0.6, lw=0)
        ax.axhline(0, color="#52514e", lw=0.8)
        m06 = allm[(allm.label == best06.get(typ, "nb06 mono, literature T_eff")) & (allm.type == typ)]
        sub = bay[(bay.label == label) & (bay.type == typ)]
        for k, (patient, structure) in enumerate(structs):
            dx = 3.0 * (k - (len(structs) - 1) / 2)
            g = sub[(sub.patient == patient) & (sub.structure == structure)].sort_values("t_spect_h")
            h = m06[(m06.patient == patient) & (m06.structure == structure) & m06.scan.isin(g.scan)]
            ax.plot(h.t_spect_h + dx + 1.5, 100 * h.error.clip(-CLIP, CLIP), ls="none", marker=MARKERS[patient], ms=6,
                    mfc="#fcfcfb", mec=C_06, mew=1.3)
            x = g.t_spect_h + dx
            ax.vlines(x, rel(g["TIA_p2.5"], g.TIA_ref), rel(g["TIA_p97.5"], g.TIA_ref), color=C_BAY, lw=0.8)
            ax.vlines(x, rel(g.TIA_p16, g.TIA_ref), rel(g.TIA_p84, g.TIA_ref), color=C_BAY, lw=2.2)
            for xx, yy, inside in zip(x, 100 * g.error.clip(-CLIP, CLIP), g.ref_in_post68):
                ax.plot(xx, yy, marker=MARKERS[patient], ms=7, color=C_BAY, mfc=C_BAY if inside else "#fcfcfb",
                        mew=1.5, ls="none")
        if i == 0:
            ax.set_title(typ, loc="left")
        if j == 0:
            ax.set_ylabel(f"{title}\nerror vs 4-point TIA (%)", fontsize=9)
        if i == 1:
            ax.set_xlabel("time of the SPECT (h)")
        ax.set_ylim(-100 * CLIP - 5, 100 * CLIP + 5)

handles = [Line2D([], [], color=C_BAY, marker="o", lw=2.2, label="posterior median, 68% (thick) and 95% (thin) interval"),
           Line2D([], [], color=C_06, marker="o", mfc="#fcfcfb", ls="none",
                  label="notebook 06, literature (uptake shape for kidneys, mono T_eff otherwise)"),
           Line2D([], [], color="#52514e", marker="o", ls="none", label="patient 4 (A)"),
           Line2D([], [], color="#52514e", marker="s", ls="none", label="patient 6 (B)"),
           Line2D([], [], color=C_BAY, marker="o", mfc="#fcfcfb", ls="none", label="reference outside posterior 68%"),
           plt.Rectangle((0, 0), 1, 1, color="#d9d8d3", label="reference 68% interval (median)")]
fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"Bayesian single-time-point TIA vs time of the SPECT (prior {MAIN})", x=0.01, ha="left")
fig.tight_layout(rect=(0, 0.09, 1, 1))
fig.savefig(KIN / "bayes_stp_error_vs_time.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Absorbed dose with intervals

Posterior TIA quantiles × the notebook 04 factor (hybrid map, VSV) of each structure give the mean absorbed dose in Gy with 68% and 95% intervals. Reference: the hybrid-map dose of notebook 04, with the 68% interval of the notebook 02 TIA. Four schemes:
- variant 1 with the SPECT at about 100 h;
- variant 2 at about 30 h and at about 100 h ($\sigma_a$ = 10%);
- the Bayesian two-time-point estimate.

All under P2. For lesions, the partial-volume uncertainty (10-20%) is not included; it applies to the reference as well.

In [ ]:
SCHEMES = [(f"V1 [{MAIN}]", "scan3", "V1: SPECT ~100 h", "#2a78d6"),
           (f"V2 sa=0.10 [{MAIN}]", "scan2", "V2: 3.7 h amplitude + SPECT ~30 h", "#eb6834"),
           (f"V2 sa=0.10 [{MAIN}]", "scan3", "V2: 3.7 h amplitude + SPECT ~100 h", "#1baf7a"),
           (f"2TP Bayes [{MAIN}]", "scan3", "2TP: SPECT ~30 h + ~100 h", "#eda100")]
dose_rows = []
for label, scan, name, _ in SCHEMES:
    s = bay[(bay.label == label) & (bay.scan == scan)].join(ref[["Gy_per_MBq_h", "Gy_per_MBq_h_MC_CT", "D_ref_Gy"]],
                                                            on=["patient", "structure"])
    for q in ["TIA_p2.5", "TIA_p16", "TIA_median", "TIA_p84", "TIA_p97.5"]:
        s["D" + q[3:] + "_Gy"] = s[q] * s.Gy_per_MBq_h
    s["D_median_MC_CT_Gy"] = s.TIA_median * s.Gy_per_MBq_h_MC_CT
    s["scheme_name"] = name
    dose_rows.append(s)
dose = pd.concat(dose_rows, ignore_index=True)
dose["D_ref_p16_Gy"] = dose.ref_p16 * dose.Gy_per_MBq_h
dose["D_ref_p84_Gy"] = dose.ref_p84 * dose.Gy_per_MBq_h
dose["dose_cell"] = [f"{r.D_median_Gy:.2f} [{r.D_p16_Gy:.2f}-{r.D_p84_Gy:.2f}] ({r['D_p2.5_Gy']:.2f}-{r['D_p97.5_Gy']:.2f})"
                     for _, r in dose.iterrows()]
dtab = dose.pivot_table(index=["patient", "structure"], columns="scheme_name", values="dose_cell", aggfunc="first")
dtab = dtab[[n for _, _, n, _ in SCHEMES]]
dtab.insert(0, "reference (nb04 hybrid) Gy [68%]",
            [f"{ref.loc[i, 'D_ref_Gy']:.2f} [{ref.loc[i, 'TIA_p16'] * ref.loc[i, 'Gy_per_MBq_h']:.2f}-"
             f"{ref.loc[i, 'TIA_p84'] * ref.loc[i, 'Gy_per_MBq_h']:.2f}]" for i in dtab.index])
print("cells: posterior median [68%] (95%), Gy")
dtab

In [ ]:
order = ref.sort_values(["type", "D_ref_Gy"]).index
fig, ax = plt.subplots(figsize=(8.5, 8))
for i, (patient, structure) in enumerate(order):
    r = ref.loc[(patient, structure)]
    ax.fill_between([r.TIA_p16 * r.Gy_per_MBq_h, r.TIA_p84 * r.Gy_per_MBq_h], i - 0.42, i + 0.42, color="#d9d8d3", lw=0)
    ax.plot([r.D_ref_Gy] * 2, [i - 0.42, i + 0.42], color="#52514e", lw=1.2)
    for k, (label, scan, name, c) in enumerate(SCHEMES):
        g = dose[(dose.scheme_name == name) & (dose.patient == patient) & (dose.structure == structure)].iloc[0]
        y = i + 0.27 - 0.18 * k
        ax.plot([g["D_p2.5_Gy"], g["D_p97.5_Gy"]], [y, y], color=c, lw=0.8)
        ax.plot([g.D_p16_Gy, g.D_p84_Gy], [y, y], color=c, lw=2.4)
        ax.plot(g.D_median_Gy, y, "o", ms=5.5, color=c, mfc=c if g.ref_in_post68 else "#fcfcfb", mew=1.3)
ax.set_xscale("log")
ax.set_xticks([1, 2, 5, 10, 20, 50], ["1", "2", "5", "10", "20", "50"])
ax.set_yticks(range(len(order)), [f"{SNMMI_LABEL[p]} {s}" for p, s in order])
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.set_xlabel("mean absorbed dose (Gy), log scale")
handles = [Line2D([], [], color=c, marker="o", lw=2.4, label=name) for _, _, name, c in SCHEMES]
handles += [Line2D([], [], color="#52514e", lw=1.2, label="reference dose (nb04 hybrid, VSV)"),
            plt.Rectangle((0, 0), 1, 1, color="#d9d8d3", label="reference 68% interval")]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.45, -0.07), ncol=2, frameon=False, fontsize=8.5)
ax.set_title("Posterior mean dose: 68% (thick) and 95% (thin) intervals; open marker: reference outside 68%",
             loc="left", fontsize=9.5)
fig.tight_layout()
fig.savefig(KIN / "bayes_stp_dose_intervals.png", dpi=150, bbox_inches="tight")
plt.show()

## Findings

Numbers from the tables above. Setup: 13 structures, reference = notebook 02 4-point TIA, main prior P2, flat prior on $\log A_0$ unless stated.

1. **Washout prior and patient 4 lesions.**
   - Parametrising on $\lambda_\mathrm{bio}$ with a log-normal (P1) does not give mass near $T_\mathrm{phys}$. The lesion prior mass above 130 h is 0.18%, against 0.70% for the truncated P0. The slow-side t tail (P2) gives 3.1% (0.66% above 150 h).
   - With one SPECT the effect on patient 4 lesions is small, because a single point carries no shape information: the posterior median $T_\mathrm{eff}$ stays at the prior median (77 h) under all priors.
     - Lesion2: -18.9% (P0), -18.0% (P1), -17.5% (P2) at about 100 h; at the last scan -11.0%, -9.9%, -9.5%, and inside the 68% interval under P1 and P2 only.
     - Lesion1 at about 100 h: -5.6% (P0) and -4.1% (P2).
   - The heavy tail matters when the data inform the shape. With two SPECTs, patient 4 lesion2 reaches a posterior $T_\mathrm{eff}$ of 112 h.
   - Cost of P2 for the SPECT at about 100 h, compared with P0:
     - median |error| 3.8% vs 3.9%;
     - 68% half-width 11.7% vs 10.8%, 95% half-width 25.2% vs 21.7%;
     - 68% and 95% coverage are equal, 12/13 and 13/13.
2. **The flat prior on $A_0$ should not be used.** Under P0 it biases the TIA upward: +2-8% at 100 h, ×1.72 for the patient 6 kidneys at 193 h. Under a heavy-tailed washout prior the posterior is improper; this was found with a symmetric $t_3$ (section 2).
3. **Variant 1 is method (3) of notebook 06 plus an interval.**
   - The kidney plug-in is identical to notebook 06. The posterior median over the plug-in is 0.997-1.048 at scans 2-4.
   - SPECT at about 100 h: median |error| 3.8% (max 17.5%). 68% half-width 11.7%, 95% half-width 25.2%. Coverage 12/13 at 68% and 13/13 at 95%. The only 68% miss is patient 4 lesion2 (-17%); patient 6 left kidney (+13%) is now just inside, with the slower kidney prior.
   - Coverage over the four scan times: 10, 11, 12, 13 of 13 at 68%; 12, 12, 13, 13 at 95%. The intervals are calibrated or slightly conservative, as expected because the reference shares the measured point.
4. **The early amplitude does not help at about 100 h, nor with the oracle uptake prior at any time.**
   - Gain in median |error| from adding the 3.7 h amplitude ($\sigma_a$ = 10%), all structures, at about 30 h / about 100 h:
     - generic uptake prior: -0.5 / -1.1 points;
     - structure-specific: +1.2 / -2.5;
     - oracle: -1.1 / -1.1.
   - For liver and lesions alone the gain is between -1.6 and +1.4 points.
   - **Kidneys at about 30 h are the exception, and they appeared with the 16-kidney prior:** the gain is +3.6 points under the generic and structure-specific priors (median |error| 9.1 to 5.5%), which also drives the +1.2 above, and -0.9 under the oracle. At about 100 h it is -3.6, -3.6 and -1.3. Because it vanishes with the correct uptake rate, we read the gain as a compensation between uptake and washout prior errors, not as information on the washout. This is an interpretation, from 4 kidneys of 2 patients.
   - With the oracle there is no gain, so the limitation is **intrinsic**:
     - At about 100 h, the single-point factor $e^{\lambda t}/\lambda$ is stationary in $\lambda$ at $t = 1/\lambda = 1.44\,T_\mathrm{eff}$ (74-111 h for $T_\mathrm{eff}$ = 51-77 h). This is the reason behind the Hänscheid method, and it is why extra information on the washout adds little there.
     - At about 30 h, the 3.7 h and 30 h points both sit in the uptake or early washout phase and do not constrain the late washout.
   - The amplitude does tighten the upper tail: the 95% half-width at 100 h drops from 25.2% to 19.2%, because the 3.7 h point excludes slow-uptake shapes that a single late point cannot rule out. Example, patient 6 left kidney: 95% dose interval 3.2-8.5 Gy with the SPECT alone, 3.2-4.6 Gy with the amplitude.
5. **Two time points (about 30 h + about 100 h).**
   - Classical mono-exponential: median |error| 2.9% (max 11.4%).
   - Bayesian: 4.3% (max 20.8%), 68% half-width 9.4%, coverage 12/13 at 68% and 13/13 at 95%.
   - The classical fit wins on patient 4 lesion2 (-2.3% vs -20.8%). Its $T_\mathrm{eff}$ is clamped to $T_\mathrm{phys}$ exactly as in the reference fit (same data, same bound). The prior still pulls the Bayesian estimate to 112 h.
   - Compared with one SPECT at about 100 h, the second SPECT mostly narrows the intervals (95% half-width 19.5% vs 25.2%); the median error is similar.
6. **Dose (section 8).** The posterior doses with the SPECT at about 100 h agree with the notebook 04 hybrid reference within the intervals, except patient 4 lesion2:
   - kidneys 2.71 [2.37-3.35] Gy vs 2.74 Gy (patient 4 left), 4.62 [4.06-5.62] vs 4.71 Gy (patient 6 right);
   - patient 6 lesion2 38.0 [34.0-42.9] vs 34.5 Gy;
   - patient 4 lesion2 29.4 [26.3-33.2] vs 35.6 [25.4-41.2] Gy.

   Monte Carlo in the patient CT instead of VSV would lower all doses by 3-7% (notebook 05). Partial volume is not included.

**Implications for the project.**
- On Deep Blue, an early amplitude measurement does not improve the TIA, even with perfect knowledge of the uptake rate.
- The pre-therapy information can help only if it constrains the **kinetics**, mainly the washout, with an honest uncertainty, including a tail toward $T_\mathrm{phys}$ for lesions. Where it could matter most is at SPECT times away from about $1.44\,T_\mathrm{eff}$ (e.g. 24-48 h), where the single-point error is dominated by the shape. This is a hypothesis to test, not a result of these two patients.
- The key experiment therefore moves to the PSMA virtual cohort, where the true kinetics are known: Bayesian update with a PET-derived kinetic prior vs the population prior, as a function of SPECT time, judged on error and on 68/95% coverage.

**Limitations.**
- Two patients and 13 structures; the SPECT point is shared with the reference. The kidney prior rests on 16 kidneys of 8 patients.
- Excluding Devasia patient 4 changed the kidney prior (median $T_\mathrm{eff}$ 51 to 55 h) and, through the generic uptake prior, all structures at early scans (up to 5% on the median TIA); at about 100 h non-kidney errors moved by at most 0.3 points.
- Assumptions in the priors: the Hänscheid range is taken as the central 95%, and the structure-specific uptake values were chosen after seeing notebook 02.
- $\sigma_a$ values are hypothetical; the 3.7 h SPECT is an idealised surrogate of a PET acquired at about 1 h with another tracer.
- One dose factor per structure (crossfire scaled with the self TIA); no partial-volume term.

## 9. Saving

Outputs in `data/processed/deepblue/kinetics/`:
- `bayes_stp_tia.csv`: one row per estimate, with label, scheme, prior, uptake prior, amplitude prior, $\sigma_a$, scans, posterior quantiles (2.5-97.5%), effective sample size, posterior $T_\mathrm{eff}$ and $\lambda_u$, reference TIA and interval, error, half-widths, coverage flags;
- `bayes_stp_dose.csv`: posterior mean dose quantiles (Gy) for the four main schemes, the VSV and MC-CT factors, and the reference dose and interval;
- `bayes_stp_priors.csv`: location and scale of every prior component, plus the kidney correlation;
- `bayes_stp_summary_by_scan.csv` (main estimators and notebook 06), `bayes_stp_coverage.csv` (all Bayesian estimators), `bayes_stp_ablation.csv`;
- `bayes_stp_error_vs_time.png`, `bayes_stp_dose_intervals.png`.

`bayes_stp_intervals.png` from the previous version is replaced by `bayes_stp_dose_intervals.png`.

In [ ]:
bay.to_csv(KIN / "bayes_stp_tia.csv", index=False)
dose.drop(columns="dose_cell").to_csv(KIN / "bayes_stp_dose.csv", index=False)
pr = [{"component": f"washout {par}", "type": typ, "log_location": loc, "log_scale": sc}
      for par, d in WASHOUT.items() for typ, (loc, sc) in d.items()]
pr += [{"component": f"uptake {u}", "type": typ, "log_location": loc, "log_scale": sc}
       for u, d in UPTAKE.items() for typ, (loc, sc) in d.items()]
pr.append({"component": "kidney corr(log lam_bio, log lam_u)", "type": "kidney", "log_location": CORR_KIDNEY, "log_scale": np.nan})
pd.DataFrame(pr).to_csv(KIN / "bayes_stp_priors.csv", index=False)
comp.to_csv(KIN / "bayes_stp_summary_by_scan.csv")
cov.to_csv(KIN / "bayes_stp_coverage.csv")
ablation.to_csv(KIN / "bayes_stp_ablation.csv", index=False)
old = KIN / "bayes_stp_intervals.png"
if old.exists():
    old.unlink()
print(len(bay), "estimates and", len(dose), "dose rows saved")